# YOLOv11n Baseline — pilliot_15k_v1

- 모델: YOLOv11n (COCO pretrained)
- 데이터: pilliot_15k_v1 (single 12k + combination 3k, 1-class: pill)
- 하이퍼파라미터: AdamW lr=0.002, 30 epochs, 640px, bs=16

In [ ]:
!pip install ultralytics -q

from google.colab import drive
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
import yaml, os, shutil, zipfile

## 1. Drive 마운트 및 경로 설정

In [ ]:
drive.mount('/content/drive')

PILLOT_ROOT   = '/content/drive/MyDrive/Conference/Pill-agent/Pillot'
DATASET_DIR   = f'{PILLOT_ROOT}/dataset'
MANIFEST_PATH = f'{DATASET_DIR}/pilliot_15k_v1_detection_manifest.csv'
ZIP_PATH      = f'{DATASET_DIR}/pilliot_15k_v1_final.zip'

assert Path(MANIFEST_PATH).exists(), f'manifest 없음: {MANIFEST_PATH}'
assert Path(ZIP_PATH).exists(),      f'zip 없음: {ZIP_PATH}'

## 2. Manifest 로드 및 스키마 확인

In [ ]:
df = pd.read_csv(MANIFEST_PATH)

required = {'image_file', 'item_seq', 'width', 'height', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h'}
missing_cols = required - set(df.columns)
assert not missing_cols, f'manifest 컬럼 누락: {missing_cols} — coco_to_yolo()의 컬럼명을 실제 스키마에 맞춰 수정 필요'

print('=== columns ===')
print(df.columns.tolist())
print('\n=== head(2) ===')
print(df.head(2).to_string())
print('\n=== shape ===')
print(df.shape)

if 'split_type' in df.columns:
    print('\n=== split_type 분포 ===')
    print(df['split_type'].value_counts())
else:
    print('\n[주의] split_type 컬럼 없음 — item_seq GroupShuffleSplit 사용 예정')

# bbox 값이 픽셀 범위인지 정규화 값인지 확인
bbox_candidates = [c for c in df.columns if any(k in c.lower() for k in ['bbox', 'box', '_x', '_y', '_w', '_h'])]
if bbox_candidates:
    print('\n=== bbox 컬럼 describe (픽셀 범위: max >> 1, 정규화: max <= 1) ===')
    print(df[bbox_candidates].describe().to_string())

## 3. zip 압축 해제

In [ ]:
shutil.rmtree('/content/raw', ignore_errors=True)

print('zip을 /content/로 복사 중...')
shutil.copy(ZIP_PATH, '/content/pilliot_15k_v1_final.zip')

print('압축 해제 중...')
with zipfile.ZipFile('/content/pilliot_15k_v1_final.zip') as zf:
    zf.extractall('/content/raw/')
os.remove('/content/pilliot_15k_v1_final.zip')

# 내부 구조 확인
raw_root = Path('/content/raw')
all_imgs = [p for p in raw_root.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
print(f'\n총 이미지 수: {len(all_imgs):,}')
print('샘플 경로 (상위 5개):')
for p in sorted(all_imgs)[:5]:
    print(f'  {p.relative_to(raw_root)}')

# 파일명 → 경로 lookup
img_lookup = {p.name: p for p in all_imgs}
print(f'\nlookup 크기: {len(img_lookup):,}')

## 4. Train / Val 분할 및 이미지 구성

In [ ]:
unique_imgs = df.drop_duplicates(subset='image_file')[['image_file', 'item_seq']]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(unique_imgs, groups=unique_imgs['item_seq']))
train_files = set(unique_imgs.iloc[train_idx]['image_file'])
val_files   = set(unique_imgs.iloc[val_idx]['image_file'])
train_df = df[df['image_file'].isin(train_files)].copy()
val_df   = df[df['image_file'].isin(val_files)].copy()
print(f'item_seq GroupShuffleSplit — train: {len(train_df):,}, val: {len(val_df):,}')

# 이미지 디렉토리 구성
shutil.rmtree('/content/images', ignore_errors=True)
missing = []
for split, subset in [('train', train_df), ('validation', val_df)]:
    dst = Path(f'/content/images/{split}')
    dst.mkdir(parents=True, exist_ok=True)
    for fname in subset['image_file'].unique():
        src = img_lookup.get(fname)
        if src:
            os.symlink(src.resolve(), dst / fname)
        else:
            missing.append(fname)
    print(f'{split}: {len(list(dst.iterdir())):,} 이미지')

if missing:
    print(f'\n[경고] lookup 미스 {len(missing)}건: {missing[:5]}')

## 5. YOLO 라벨 변환
COCO `[x, y, w, h]` (픽셀) → YOLO `[xc, yc, w, h]` (정규화)

In [ ]:
def coco_to_yolo(row):
    bx, by, bw, bh = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
    iw, ih = row['width'], row['height']
    if pd.isna(bx) or bw < 1 or bh < 1:
        return None
    xc = (bx + bw / 2) / iw
    yc = (by + bh / 2) / ih
    return f'0 {xc:.6f} {yc:.6f} {bw/iw:.6f} {bh/ih:.6f}'

def write_labels(subset_df, split):
    label_dir = Path(f'/content/labels/{split}')
    label_dir.mkdir(parents=True, exist_ok=True)
    skipped = 0
    for fname, grp in subset_df.groupby('image_file'):
        lines = []
        for _, row in grp.iterrows():
            line = coco_to_yolo(row)
            if line:
                lines.append(line)
            else:
                skipped += 1
        if lines:
            (label_dir / (Path(fname).stem + '.txt')).write_text('\n'.join(lines))
    n = len(list(label_dir.glob('*.txt')))
    print(f'{split}: {n:,} 라벨 파일 생성, {skipped} annotation 스킵')

shutil.rmtree('/content/labels', ignore_errors=True)
write_labels(train_df, 'train')
write_labels(val_df,   'validation')

## 6. 이미지-라벨 매칭 검증

In [ ]:
def validate_pairs(split):
    imgs   = {p.stem for p in Path(f'/content/images/{split}').iterdir()
              if p.suffix in {'.jpg', '.png'}}
    labels = {p.stem for p in Path(f'/content/labels/{split}').glob('*.txt')}
    no_label = imgs - labels
    no_image = labels - imgs
    print(f'[{split}] images: {len(imgs):,}, labels: {len(labels):,}')
    print(f'  라벨 없는 이미지: {len(no_label)}')
    print(f'  이미지 없는 라벨: {len(no_image)}')

validate_pairs('train')
validate_pairs('validation')

## 7. dataset.yaml 생성

In [ ]:
cfg = {
    'path':  '/content',
    'train': 'images/train',
    'val':   'images/validation',
    'nc':    1,
    'names': ['pill'],
}
with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print(open('/content/dataset.yaml').read())

## 8. 학습

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.002,
    device=0,
    project='/content/runs',
    name='yolo11n_15k_baseline',
)

## 9. 검증

In [ ]:
metrics = model.val()
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 10. 결과 Drive 저장

In [ ]:
dst = f'{PILLOT_ROOT}/runs/yolo11n_15k_baseline'
shutil.copytree(
    '/content/runs/yolo11n_15k_baseline',
    dst,
    dirs_exist_ok=True
)
print(f'저장 완료: {dst}')